# English -> Hindi Translator

<br></br>

## Author: Dr Partha Majumdar
#### ORC-ID: 0009-0002-7375-8034

<br></br>

Dataset  : cfilt/iitb-english-hindi

Model    : Helsinki-NLP/opus-mt-en-hi

Library  : transformers + datasets




# Install if needed

In [1]:
# !pip install -qq transformers==5.0.0 datasets==4.0.0 sentencepiece==0.2.1 sacrebleu==2.6.0 evaluate==0.4.6 accelerate==1.13.0 sacremoses==0.1.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 51.5 MB/s eta 0:00:00


In [2]:
import importlib.metadata as im

for pkg in [
    "transformers", "datasets", "sentencepiece", "sacrebleu", "evaluate", "accelerate", "sacremoses"
]:
    print(pkg, "==", im.version(pkg))

transformers == 5.0.0
datasets == 4.0.0
sentencepiece == 0.2.1
sacrebleu == 2.6.0
evaluate == 0.4.6
accelerate == 1.13.0
sacremoses == 0.1.1


In [ ]:
import pandas as pd
import numpy as np
import transformers
import datasets
import torch
from datasets import load_dataset, Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback,
    AutoConfig,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
import evaluate

print("Pandas version:", pd.__version__)
print("Numpy version:", np.__version__)
print("Hugging Face Transformers version:", transformers.__version__)
print("Hugging Face Datasets version:", datasets.__version__)
print("Evaluate version:", evaluate.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("CUDNN version:", torch.backends.cudnn.version())
print("GPU name:", torch.cuda.get_device_name(0))
print("GPU memory:", torch.cuda.get_device_properties(0).total_memory)
print("GPU memory allocated:", torch.cuda.memory_allocated(0))
print("GPU memory reserved:", torch.cuda.memory_reserved(0))
print("GPU memory free:", torch.cuda.memory_reserved(0) - torch.cuda.memory_allocated(0))
print("GPU memory max:", torch.cuda.max_memory_allocated(0))
print("GPU memory max reserved:", torch.cuda.max_memory_reserved(0))
print("GPU memory max free:", torch.cuda.max_memory_reserved(0) - torch.cuda.max_memory_allocated(0))
print("All Libraries Loaded")

Pandas version: 2.2.2
Numpy version: 2.0.2
Hugging Face Transformers version: 5.0.0
Hugging Face Datasets version: 4.0.0
Evaluate version: 0.4.6
CUDA available: True
CUDA version: 12.8
CUDNN version: 91002
GPU name: Tesla T4
GPU memory: 15637086208
GPU memory allocated: 0
GPU memory reserved: 0
GPU memory free: 0
GPU memory max: 0
GPU memory max reserved: 0
GPU memory max free: 0
All Libraries Loaded


The experimental environment for this chapter was carefully configured to ensure both reproducibility and computational efficiency. The implementation was executed using Pandas 2.2.2 and NumPy 2.0.2 for data handling, alongside the Hugging Face ecosystem—Transformers 5.0.0, Datasets 4.0.0, and Evaluate 0.4.6—for model development and evaluation. The computations were accelerated using CUDA-enabled hardware, specifically an NVIDIA Tesla T4 GPU with approximately 16 GB of memory, supported by CUDA version 12.8 and cuDNN version 91002. At the start of execution, all GPU memory was unallocated, ensuring a clean and controlled runtime environment. This configuration allowed efficient fine-tuning of sequence-to-sequence models while maintaining clarity and consistency in experimentation. All required libraries were successfully loaded, confirming the readiness of the system for end-to-end implementation.

# STEP 1: Load dataset

In [ ]:
dataset = load_dataset("cfilt/iitb-english-hindi")

print("STEP 1: Load Dataset")

for split in dataset:
    print(f"{split} size:", len(dataset[split]))

total = sum(len(dataset[split]) for split in dataset)
print("Total data points:", total)

# ------------------------------------------------------------
# Use a manageable subset for demonstration / book example
# ------------------------------------------------------------
train_size = 50000 # 100000
valid_size = 5000 # 5000
test_size = 2500 # 5000

train_data = dataset["train"].shuffle(seed=42).select(range(train_size))
valid_data = dataset["validation"].shuffle(seed=42).select(range(min(valid_size, len(dataset["validation"]))))
test_data  = dataset["test"].shuffle(seed=42).select(range(min(test_size, len(dataset["test"]))))

# ------------------------------------------------------------
# Convert to DataFrame for basic cleaning
# ------------------------------------------------------------
train_df = pd.DataFrame({
    "english": [item["translation"]["en"] for item in train_data],
    "hindi": [item["translation"]["hi"] for item in train_data]
})

valid_df = pd.DataFrame({
    "english": [item["translation"]["en"] for item in valid_data],
    "hindi": [item["translation"]["hi"] for item in valid_data]
})

test_df = pd.DataFrame({
    "english": [item["translation"]["en"] for item in test_data],
    "hindi": [item["translation"]["hi"] for item in test_data]
})

# ------------------------------------------------------------
# Basic cleaning
# ------------------------------------------------------------
def clean_df(df):
    df = df.copy()
    df.dropna(inplace=True)
    df.drop_duplicates(inplace=True)

    df["english"] = df["english"].astype(str).str.strip()
    df["hindi"] = df["hindi"].astype(str).str.strip()

    # Remove empty rows
    df = df[(df["english"] != "") & (df["hindi"] != "")]

    # Filter by word count, not character count
    MAX_EN_WORDS = 40
    MAX_HI_WORDS = 45

    df = df[df["english"].apply(lambda x: len(x.split()) <= MAX_EN_WORDS)]
    df = df[df["hindi"].apply(lambda x: len(x.split()) <= MAX_HI_WORDS)]

    return df.reset_index(drop=True)

train_df = clean_df(train_df)
valid_df = clean_df(valid_df)
test_df = clean_df(test_df)

print("\nAfter cleaning:")
print("Train size:", len(train_df))
print("Valid size:", len(valid_df))
print("Test size :", len(test_df))

print("\nSample examples:")
print(train_df.head(3))

# ------------------------------------------------------------
# Convert back to Hugging Face Dataset
# ------------------------------------------------------------
dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df, preserve_index=False),
    "validation": Dataset.from_pandas(valid_df, preserve_index=False),
    "test": Dataset.from_pandas(test_df, preserve_index=False),
})

print("\nRebuilt DatasetDict:")
print(dataset)
print(dataset["train"][0])

README.md: 0.00B [00:00, ?B/s]

dataset_infos.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/190M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/85.7k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/500k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1659083 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/520 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2507 [00:00<?, ? examples/s]

STEP 1: Load Dataset
train size: 1659083
validation size: 520
test size: 2507
Total data points: 1662110

After cleaning:
Train size: 46238
Valid size: 510
Test size : 2370

Sample examples:
                                             english  \
0                                  on the intuition.   
1                                    Ceiba pentandra   
2  Have you not regarded how your Lord dealt with...   

                                               hindi  
0                                     अंतर्ज्ञान पर।  
1                                           कण्टकारी  
2  क्या तुमने देखा नहीं कि तुम्हारे आद के साथ क्य...  

Rebuilt DatasetDict:
DatasetDict({
    train: Dataset({
        features: ['english', 'hindi'],
        num_rows: 46238
    })
    validation: Dataset({
        features: ['english', 'hindi'],
        num_rows: 510
    })
    test: Dataset({
        features: ['english', 'hindi'],
        num_rows: 2370
    })
})
{'english': 'on the intuition.', 'hindi': 'अंतर्

For the purpose of this demonstration, although the original dataset contains over 1.6 million sentence pairs, a carefully cleaned and filtered subset has been used. After preprocessing, the working dataset consists of approximately 46,000 training examples, along with smaller validation and test sets. This deliberate reduction allows for faster experimentation, clearer illustration of concepts, and efficient use of available computational resources, particularly within a constrained GPU environment. It is important to note, however, that using the full dataset would significantly enhance the model's performance. A larger corpus exposes the model to greater linguistic diversity, richer contextual patterns, and broader vocabulary coverage, thereby improving translation fluency, accuracy, and generalization. Thus, while the subset serves the pedagogical purpose of this chapter effectively, scaling to the complete dataset would lead to a more robust and production-grade translation system.

# STEP 2: Load pretrained tokeniser and model

In [ ]:
MODEL_NAME = "Helsinki-NLP/opus-mt-en-hi"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

print("STEP 2: Tokeniser and Model Loaded")
print("Model name:", MODEL_NAME)

# Keep only the essential safe settings
model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id

if model.config.decoder_start_token_id is None:
    model.config.decoder_start_token_id = tokenizer.pad_token_id

model.generation_config.pad_token_id = tokenizer.pad_token_id
model.generation_config.eos_token_id = tokenizer.eos_token_id
model.generation_config.decoder_start_token_id = model.config.decoder_start_token_id

print("\nSpecial Tokens Aligned")
print("decoder_start_token_id :", model.generation_config.decoder_start_token_id)
print("eos_token_id           :", model.generation_config.eos_token_id)
print("pad_token_id           :", model.generation_config.pad_token_id)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/812k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/306M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

STEP 2: Tokeniser and Model Loaded
Model name: Helsinki-NLP/opus-mt-en-hi

Special Tokens Aligned
decoder_start_token_id : 61949
eos_token_id           : 0
pad_token_id           : 61949


The translation system in this chapter is built using the pretrained model Helsinki-NLP/opus-mt-en-hi, accessed through the Hugging Face Transformers library. This model is part of the OPUS-MT family of sequence-to-sequence neural machine translation systems, trained on large-scale parallel corpora covering multiple language pairs. It follows an encoder-decoder architecture, where the input English sentence is first converted into token embeddings, processed into a contextual representation, and then decoded into a corresponding Hindi sequence. The tokeniser and model are loaded directly from the pretrained checkpoint, ensuring that the learned vocabulary, tokenisation scheme, and embedding space remain consistent with the training distribution.

In this implementation, only the essential configuration parameters are aligned—specifically the padding, end-of-sequence, and decoder start tokens—to ensure smooth generation during inference. No structural modifications are made to the model, preserving its pretrained capabilities. This allows the focus to remain on how tokenisation and embeddings drive the transformation from one language to another, while leveraging a robust, production-grade translation model as the underlying engine.

# STEP 3: Tokenisation

In [ ]:
MAX_SOURCE_LENGTH = 64
MAX_TARGET_LENGTH = 64

def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["english"],
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
        padding=False
    )

    labels = tokenizer(
        text_target=examples["hindi"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding=False
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=["english", "hindi"]
)

print("STEP 3: Tokenisation completed")
print(tokenized_datasets)

Map:   0%|          | 0/46238 [00:00<?, ? examples/s]

Map:   0%|          | 0/510 [00:00<?, ? examples/s]

Map:   0%|          | 0/2370 [00:00<?, ? examples/s]

STEP 3: Tokenisation completed
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 46238
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 510
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2370
    })
})


In this step, the textual data is transformed into a numerical representation that the model can process. Each English sentence is tokenised into a sequence of input IDs, along with an attention mask that indicates which tokens are meaningful and which positions are padding. Similarly, the corresponding Hindi sentences are tokenised into target sequences, which are assigned as labels for supervised training. The maximum sequence lengths for both source and target are set to 64 tokens, ensuring a consistent upper bound while allowing shorter sequences to remain unpadded at this stage. Through the map operation, this transformation is efficiently applied across the entire dataset in batches, replacing the original text columns with structured numerical features. The result is a fully tokenised dataset where language has been reduced to sequences of indices—precisely the form required for the embedding layer to convert them into vectors for subsequent processing by the model.

# STEP 4: Data collator

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

The data collator dynamically prepares batches during training by applying padding to ensure that all sequences in a batch have the same length. It uses the tokeniser to handle padding correctly and aligns inputs and labels in a format suitable for the sequence-to-sequence model, enabling efficient batch processing without unnecessary pre-padding of the entire dataset.

# STEP 5: Metric

In [ ]:
import numpy as np
import evaluate

bleu_metric = evaluate.load("sacrebleu")

def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]
    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    if isinstance(preds, tuple):
        preds = preds[0]

    # Replace ignored labels
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    # Safety: keep ids inside tokenizer vocab range
    vocab_limit = tokenizer.vocab_size - 1
    preds = np.clip(preds, 0, vocab_limit)
    labels = np.clip(labels, 0, vocab_limit)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)
    result = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)

    return {"bleu": result["score"]}

In this step, a custom evaluation function is defined to measure the quality of the model's translations using the BLEU metric, a standard benchmark for machine translation. The model's predictions and reference labels, initially represented as token IDs, are first processed to handle ignored values (marked as -100 during training) and to ensure all token indices remain within the valid vocabulary range. These numerical sequences are then decoded back into human-readable text using the tokenizer. A light post-processing step removes extraneous whitespace and formats the references appropriately for evaluation. Finally, the sacrebleu metric is applied to compare the generated translations against the ground truth, producing a BLEU score that quantitatively reflects how closely the model's outputs align with the expected translations.

# STEP 6: Training arguments

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./en_hi_translation_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=10,
    predict_with_generate=True,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    greater_is_better=True,
    report_to="none"
)

The training process is governed by a carefully chosen set of arguments that balance learning efficiency, stability, and computational feasibility. The model outputs are stored in a designated directory, ensuring that intermediate checkpoints and the final trained model can be accessed and reused. Evaluation and model saving are both performed at the end of each epoch, allowing the training process to be monitored systematically and ensuring that the model's performance on unseen data is regularly assessed. Logging is carried out at fixed step intervals, providing visibility into the training dynamics without overwhelming the output with excessive detail.

The learning rate is set to a small value, reflecting the fact that the model is being fine-tuned rather than trained from scratch. Batch sizes for both training and evaluation are chosen to efficiently utilise the available GPU memory, while gradient accumulation is employed to simulate a larger effective batch size without exceeding memory constraints. Weight decay is introduced as a regularisation mechanism, helping to prevent overfitting by discouraging overly large parameter values. The number of training epochs is defined as an upper limit, with the understanding that the model may converge earlier depending on its learning behaviour.

A key feature of this configuration is the use of sequence generation during evaluation, enabling the model to produce actual translated sentences rather than merely computing token-level losses. This allows performance to be assessed using the BLEU metric, which is specified as the primary criterion for selecting the best model. By enabling the loading of the best-performing model at the end of training, the system ensures that the final output reflects the highest observed translation quality. Mixed-precision training is activated to accelerate computation and reduce memory usage on compatible hardware, while external reporting is disabled to maintain a focused and self-contained experimental setup.

# STEP 7: Trainer

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

The Seq2SeqTrainer orchestrates the entire training process for the translation model by integrating the encoder-decoder architecture with the prepared data and training configuration. In this setup, the model follows an encode-decode paradigm: the encoder first processes the tokenised English input sequence, transforming it into a contextual representation in vector space, and the decoder then generates the corresponding Hindi sequence token by token based on this representation. The trainer manages how batches are fed into this pipeline, how gradients are computed and updated, and how performance is evaluated on the validation set. It uses the tokenizer for consistent preprocessing, the data collator for efficient batching, and the custom metric function to assess translation quality using BLEU. Additionally, early stopping is incorporated to halt training when no further improvement is observed, ensuring that the model remains well-generalised without unnecessary overtraining.

# STEP 8: Fine-tune

In [ ]:
print("STEP 8: Training starts")
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


STEP 8: Training starts


Epoch,Training Loss,Validation Loss,Bleu
1,5.933680,3.760930,9.130775
2,5.392896,3.668272,9.644557
3,5.049029,3.605609,9.757516
4,4.790298,3.581901,10.893869
5,4.511342,3.568054,10.787642
6,4.413052,3.558308,11.039588
7,4.268134,3.554778,10.785475
8,4.074582,3.559836,11.062791
9,4.034314,3.558995,11.216094
10,4.000938,3.559039,10.793749


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_positions.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].


TrainOutput(global_step=14450, training_loss=4.704948739972494, metrics={'train_runtime': 2788.5502, 'train_samples_per_second': 165.814, 'train_steps_per_second': 5.182, 'total_flos': 5162616105467904.0, 'train_loss': 4.704948739972494, 'epoch': 10.0})

The training process presented here is not a case of building a translation system from scratch, but rather an instance of fine-tuning a pretrained model. The model, Helsinki-NLP/opus-mt-en-hi, has already been trained on large-scale multilingual parallel corpora and therefore possesses a substantial prior understanding of both English and Hindi. What is being done in this chapter is to adapt this existing knowledge to the specific subset of data being used. This is reflected in the relatively rapid convergence of the training process, where both training and validation loss decrease steadily across epochs. Instead of learning language structure from the ground up, the model is refining its internal representations—adjusting its embedding space and transformation layers to better align with the provided data.

The evaluation of the model is carried out using the BLEU (Bilingual Evaluation Understudy) score, which is a standard metric for machine translation. BLEU measures how closely the model's generated translations match reference translations by comparing overlapping sequences of words (n-grams). A higher BLEU score indicates better alignment with the expected output, although it is important to understand that BLEU captures statistical similarity rather than perfect semantic equivalence. In this experiment, the BLEU score improves from approximately 9.1 in the first epoch to around 11.2 at its peak. While this may appear modest in absolute terms, it is consistent with the constrained training setup—particularly the use of a reduced dataset and limited epochs. More importantly, the steady improvement across epochs demonstrates that the model is successfully adapting to the data and learning meaningful cross-lingual mappings.

# STEP 9: Evaluate

In [ ]:
print("STEP 9: Evaluation on test set")
test_results = trainer.evaluate(tokenized_datasets["test"])
print(test_results)

STEP 9: Evaluation on test set


{'eval_loss': 3.3228938579559326, 'eval_bleu': 12.663677676709709, 'eval_runtime': 162.4597, 'eval_samples_per_second': 14.588, 'eval_steps_per_second': 0.917, 'epoch': 10.0}


# STEP 10: Save model

In [ ]:
trainer.save_model("./en_hi_translation_model")
tokenizer.save_pretrained("./en_hi_translation_model")

print("STEP 10: Model saved")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

STEP 10: Model saved


# STEP 11: Inference function

In [ ]:
def translate_english_to_hindi(text, model, tokenizer, max_new_tokens=64):
    model.eval()

    device = model.device

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=64
    )

    # Move all tokenizer outputs to the model device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        num_beams=4,
        early_stopping=True
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# STEP 12: Test translations

In [ ]:
test_sentences = [
    "Hello.",
    "How are you?",
    "I am learning machine learning.",
    "This book is about word embeddings.",
    "Language models begin with token embeddings.",
    "We are studying translation.",
    "She is reading a book.",
    "Thank you very much."
]

print("STEP 12: Sample translations\n")
for sent in test_sentences:
    translated = translate_english_to_hindi(sent, model, tokenizer)
    print("EN:", sent)
    print("HI:", translated)
    print("-" * 70)

STEP 12: Sample translations

EN: Hello.
HI: हैलो।
----------------------------------------------------------------------
EN: How are you?
HI: तुम कैसे हो?
----------------------------------------------------------------------
EN: I am learning machine learning.
HI: मैं मशीन सीखने के लिए तैयार हूं।
----------------------------------------------------------------------
EN: This book is about word embeddings.
HI: इस पुस्तक में इस शब्द के बारे में चर्चा की गई है।
----------------------------------------------------------------------
EN: Language models begin with token embeddings.
HI: भाषा मॉडलों के प्रतीकों के साथ प्रारंभ होता है।
----------------------------------------------------------------------
EN: We are studying translation.
HI: हम अनुवाद पढ़ रहे हैं।
----------------------------------------------------------------------
EN: She is reading a book.
HI: वह एक किताब पढ़ रही है।
----------------------------------------------------------------------
EN: Thank you very much.
HI: बहुत ब

The sample translations provide a meaningful qualitative insight into the model's behaviour and learning. Even with a modest BLEU score, the outputs demonstrate that the system has acquired a functional understanding of cross-lingual mapping. Simple sentences such as “Hello” and “How are you?” are translated accurately and naturally, indicating that the model has retained strong foundational knowledge from pretraining. More complex sentences show partial correctness, where the structure and intent are preserved, even if some phrasing is imperfect. For instance, “This book is about word embeddings” is translated into a grammatically valid Hindi sentence that captures the idea of discussion, though the phrase “word embeddings” is not rendered precisely.

A particularly interesting observation is the model's handling of gender agreement in Hindi. In the sentence “She is reading a book,” the model correctly produces “वह एक किताब पढ़ रही है,” using the feminine verb form “रही है.” This indicates that the model is not merely translating word by word, but is applying learned grammatical patterns embedded in its internal representations. At the same time, there are instances where the model makes semantic substitutions, such as translating “I am learning machine learning” into a sentence that conveys preparation rather than learning. These deviations highlight the limitations imposed by the reduced dataset and limited fine-tuning.

Overall, the outputs reflect a system that has successfully internalised key linguistic patterns—syntax, agreement, and basic semantics—while still exhibiting gaps in domain-specific precision. This reinforces an important point: even when quantitative metrics are modest, qualitative analysis reveals that the embedding-driven model is genuinely constructing meaning across languages rather than simply memorising phrases.

# STEP 13: Inspect embedding layers

In [ ]:
print("STEP 13: Embedding layer shapes")

# Marian stores shared embeddings
shared_embeddings = model.model.shared.weight
print("Shared embedding matrix shape:", shared_embeddings.shape)

# Encoder / decoder embeddings
encoder_embeddings = model.model.encoder.embed_tokens.weight
decoder_embeddings = model.model.decoder.embed_tokens.weight

print("Encoder embedding matrix shape:", encoder_embeddings.shape)
print("Decoder embedding matrix shape:", decoder_embeddings.shape)

# ------------------------------------------------------------
# Show a few token embeddings
# ------------------------------------------------------------
sample_tokens = ["hello", "language", "model", "translation"]

print("\nSample token embeddings from tokenizer vocabulary:")
for tok in sample_tokens:
    token_ids = tokenizer(tok, add_special_tokens=False)["input_ids"]
    print(f"\nToken: {tok}")
    print("Token IDs:", token_ids)

    for tid in token_ids:
        vector = shared_embeddings[tid].detach().cpu().numpy()
        print(f"ID {tid} -> {vector[:10]} ...")

STEP 13: Embedding layer shapes
Shared embedding matrix shape: torch.Size([61950, 512])
Encoder embedding matrix shape: torch.Size([61950, 512])
Decoder embedding matrix shape: torch.Size([61950, 512])

Sample token embeddings from tokenizer vocabulary:

Token: hello
Token IDs: [39915]
ID 39915 -> [-0.02698568  0.02463055 -0.02686038  0.01705558 -0.01421272  0.03075512
  0.02389886  0.03926139  0.00070265 -0.01357831] ...

Token: language
Token IDs: [1567]
ID 1567 -> [-0.0201219   0.04226342  0.01915283 -0.03633476 -0.05152645  0.06290326
  0.00309233 -0.01231997 -0.01619053 -0.02246944] ...

Token: model
Token IDs: [6644]
ID 6644 -> [-0.04471652  0.0123596  -0.03170924 -0.01465295 -0.02993963  0.02379213
  0.01633605  0.00743726 -0.03240526 -0.04560909] ...

Token: translation
Token IDs: [5753]
ID 5753 -> [-0.04294377  0.05667596 -0.01461741 -0.02396675 -0.02185631  0.04875824
 -0.00565193  0.02412932 -0.05095793 -0.01532234] ...


This final step brings the entire journey of the book to its natural conclusion. The embedding matrices—shared across encoder and decoder—map a vocabulary of over 61,000 tokens into 512-dimensional vectors, forming the foundational layer upon which all higher-level language understanding is built. Each token, whether “hello,” “language,” or “translation,” is no longer treated as a discrete symbol but as a point in a continuous vector space, where meaning is encoded numerically. The fact that the same embedding space is used across the model highlights a central theme of this book: language processing begins with representation. All transformations—contextual understanding, sequence modeling, and translation—operate on these vectors. Thus, what started in the early chapters as simple word embeddings now culminates in a full-fledged system where token embeddings serve as the entry point to complex linguistic behavior. This reinforces the core idea that embeddings are not an isolated concept, but the very first layer of modern language models, enabling everything that follows.

# STEP 14: Load saved model later

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("./en_hi_translation_model")
model = AutoModelForSeq2SeqLM.from_pretrained("./en_hi_translation_model")

print(translate_english_to_hindi("I am studying embeddings.", model, tokenizer))

Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


मैं एम्बेडिंग का अध्ययन कर रहा हूँ।


What we have built here is not merely a translation system—it is a living demonstration of understanding emerging from representation. The fact that the model correctly processes a sentence like “I am studying embeddings” and produces a meaningful Hindi translation shows that it has internalised not just common conversational patterns, but also abstract, technical vocabulary. This is significant, because “embeddings” is not a trivial everyday word; it belongs to a conceptual domain, and yet the system is able to situate it within the broader linguistic structure. This reflects the power of token embeddings and the shared vector space we have explored throughout the book. We have successfully moved from first principles to a working system that embodies those principles, and in doing so, we have demonstrated that when language is transformed into vectors, even complex ideas can be carried across languages with coherence and intent.